# Start here: capstone N-PORT data
Run `python run.py` first. This notebook reads the resulting database; it does not require a server.

The default universe has 59 series, including benchmarks. PRIMARY is the core 49-fund analysis group. Read README.md before using this data for prediction.


In [ ]:
from pathlib import Path
import sqlite3
import pandas as pd
import research

DB = Path("output/nport.sqlite")
assert DB.exists(), "Run python run.py first, or set DB to your output database"
con = sqlite3.connect(DB)


## 1. Check coverage before analyzing results

In [ ]:
coverage = pd.read_sql_query("SELECT * FROM coverage", con)
display(coverage.groupby("group").agg(funds=("series_id", "size"), observed=("report_count", lambda x: (x > 0).sum())))
display(coverage[["series_id", "fund_name", "report_count", "first_report_date", "last_report_date"]])


## 2. Your main fund-quarter table
One row per exact report date, preserving different fiscal calendars. The normalized risk columns retain reported signs; validate the convention before interpreting economic duration.

In [ ]:
fund_quarter = pd.read_sql_query("SELECT * FROM fund_quarter WHERE universe_group = 'PRIMARY'", con)
display(fund_quarter[["series_id", "fund_name", "report_date", "net_assets", "usd_rate_sensitivity_years_signed", "ig_spread_sensitivity_years_signed"]].head(15))
risk = pd.read_sql_query("SELECT * FROM risk_normalized_latest", con)
display(risk.head())


## 3. Returns and flows have different keys
Returns are by share class. Flows are by fund. Do not join flows to every class and then sum them.

In [ ]:
returns = pd.read_sql_query("SELECT * FROM monthly_returns_latest", con)
flows = pd.read_sql_query("SELECT * FROM monthly_flows_latest", con)
class_coverage = returns.groupby(["series_id", "class_id"], dropna=False).agg(months=("month_end", "nunique"), first_month=("month_end", "min"), last_month=("month_end", "max")).reset_index()
display(class_coverage.head(20))


### Optional reference-class selection
Create `reference_classes.csv` with columns `series_id,class_id`, using verified class identities. Only include the funds you have mapped. Use `__NO_CLASS__` only for genuinely classless reported returns. Do not choose the first class alphabetically. This simple helper supports one stable class per series; if a class changes historically, build a separate dated mapping.

In [ ]:
mapping_path = Path("reference_classes.csv")
if mapping_path.exists():
    mapping = pd.read_csv(mapping_path, dtype=str)
    fund_month = research.reference_class_panel(mapping, DB)
    display(fund_month.head())
else:
    print("All class returns are ready. Add reference_classes.csv when class selection is verified.")


## 4. Query one fund without loading the entire holdings history

In [ ]:
series_id = "S000053068"  # Vanguard Core Bond Fund, from the supplied universe
latest_report = con.execute("SELECT MAX(report_date) FROM holdings_latest WHERE series_id=?", (series_id,)).fetchone()[0]
holdings = pd.read_sql_query("SELECT * FROM holdings_enriched_latest WHERE series_id=? AND report_date=?", con, params=(series_id, latest_report))
display(holdings.head())
print("Positions:", len(holdings), "Reported weight sum:", holdings.PERCENTAGE.sum(min_count=1))
# This is not a duration estimate. Derivative market values can be very small.


## 5. Read derivative details without multiplying positions

In [ ]:
swaps = pd.read_sql_query("SELECT * FROM nonforeign_exchange_swap_latest WHERE series_id=? AND report_date=?", con, params=(series_id, latest_report))
display(swaps.head())
# Includes different swap types. Inspect reference instruments/payment legs before labeling an interest-rate swap.
# Never infer net duration direction from pay-fixed vs receive-fixed NOTIONAL totals alone.


## 6. Select historical snapshots before joining outcomes
Availability defaults to a lag proxy, not a verified public timestamp. The function filters eligible amendments first and returns age in days. It does not claim a fully point-in-time-certified history.

In [ ]:
asof = research.snapshots_asof("2026-03-31", DB)
display(asof[["series_id", "report_date", "filing_date", "available_date_assumed", "availability_basis", "snapshot_age_days"]].head(15))


## 7. Prepare the external-data crosswalk
Keep ID mappings separate from the raw outputs. Fill from verified sources. Use temporal validity where mappings change. This cell does not overwrite an existing mapping.

In [ ]:
crosswalk_file = Path("external_id_crosswalk.csv")
if not crosswalk_file.exists():
    template = coverage[["series_id", "fund_name"]].copy()
    for col in ["class_id", "bloomberg_fund_id", "crsp_fundno", "valid_from", "valid_to", "source"]:
        template[col] = ""
    template.to_csv(crosswalk_file, index=False)
print("Crosswalk:", crosswalk_file)
# Future fee merge: series_id + class_id + effective month after publication/effective-date handling.
# Future factor merge: month_end; retain each factor's own release/publication timing for forecasts.


## 8. Optional Parquet export in bounded chunks
Run this only if you want separate Parquet holdings files. Install pyarrow first. SQLite remains the source database. Use a new destination folder to avoid stale files.

In [ ]:
EXPORT_PARQUET = False
if EXPORT_PARQUET:
    destination = Path("holdings_parquet")
    destination.mkdir(exist_ok=False)
    for i, chunk in enumerate(pd.read_sql_query("SELECT * FROM holdings_enriched_latest", con, chunksize=100_000)):
        chunk.to_parquet(destination / f"part-{i:05d}.parquet", index=False)


## 9. Review diagnostics
No dates, holdings or returns are forward-filled. Missing periods and outliers need investigation; the pipeline never deletes them automatically.

In [ ]:
display(pd.read_sql_query("SELECT * FROM coverage_gaps", con).head(20))
display(pd.read_sql_query("SELECT series_id,report_date,usd_rate_sensitivity_years_signed,risk_review FROM risk_review_flags WHERE risk_review <> 'WITHIN_MAGNITUDE_BOUNDS_SIGN_UNVERIFIED'", con))
con.close()
